# PolyClustX: from feature views to attributed graphs

Compare several methods on two datasets, add graph relations, inspect embeddings and loss curves, tune hyperparameters, and export results.

**This notebook has not been executed.** Run cells in order after installing PolyClustX in the selected Jupyter kernel. Iris and Wine ship with scikit-learn; no dataset download is needed. Feature groups are illustrative views of the same modality, not genuinely separate acquisition modalities. Short CPU schedules demonstrate the API, not benchmark-quality settings.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, load_wine
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler

from polyclustx.data import MultiViewDataset
from polyclustx.experiment import (
    Experiment, GridSearchExperiment, RandomSearchExperiment, OptunaSearchExperiment,
)
from polyclustx.metrics import clustering_accuracy, nmi_score, ari_score
from polyclustx.models import MultiViewKMeans, LMGEC, IDEC
from polyclustx.visualization import plot_clusters, plot_loss_comparison

OUTPUT = Path("experiments/polyclustx-feature-to-graph")
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 42
RUNS = 2
metrics = [clustering_accuracy, nmi_score, ari_score]

## 1. Prepare two datasets

Split each feature matrix into two column groups while preserving row order. Scaling uses all samples because this is a transductive clustering demonstration, not held-out prediction. Labels are supplied only for evaluation. Both datasets have three reference classes; we use that count here, but you can choose another K.

In [ ]:
def feature_views(loader, name):
    source = loader()
    groups = np.array_split(np.arange(source.data.shape[1]), 2)
    views = [
        StandardScaler().fit_transform(source.data[:, group]).astype(np.float32)
        for group in groups
    ]
    return MultiViewDataset(
        views, labels=source.target, name=name,
        view_names=["feature-group-1", "feature-group-2"],
        modalities=["tabular", "tabular"],
    )

datasets = {
    "iris-features": feature_views(load_iris, "iris-features"),
    "wine-features": feature_views(load_wine, "wine-features"),
}
pd.DataFrame([
    {"dataset": name, "samples": len(data.Xs[0]),
     "view dimensions": [x.shape[1] for x in data.Xs], "classes": data.n_clusters}
    for name, data in datasets.items()
])

## 2. Compare methods with repeated experiments

The runner prints configuration and metric tables and saves one detailed JSON report. Repeated runs use different seeds. LMGEC's two-dimensional embedding is valid even for the two-feature Iris views. Models that ignore graphs can still be used with feature-only data.

In [ ]:
models = [
    MultiViewKMeans(n_clusters=3, n_init=3, max_iter=25, random_state=SEED, device="cpu"),
    LMGEC(n_clusters=3, embedding_dim=2, max_iter=15, random_state=SEED, device="cpu"),
    IDEC(n_clusters=3, embedding_dim=4, hidden_dims=(16,), pretrain_epochs=5,
         max_iter=5, batch_size=64, n_init=3, random_state=SEED, device="cpu"),
]
comparison = Experiment(
    datasets=datasets, models=models, metrics=metrics, runs=RUNS,
    save_path=str(OUTPUT / "feature-comparison"),
).run()
print(comparison.config_table())
print(comparison.results_tables())
print("Detailed report:", comparison.json_path)

## 3. Visualize a representative fit

Fit one explicit representative model for the projection panels. A shared PCA projection makes the true/predicted label comparison meaningful. Clusters are identifiers, not class names: their numeric labels need not match. The loss comparison uses run 1 from each recorded configuration. Normalized losses compare trajectory shapes, not objective quality across different algorithms.

In [ ]:
iris = datasets["iris-features"]
representative = iris.fit(
    LMGEC(n_clusters=3, embedding_dim=2, max_iter=15, random_state=SEED, device="cpu")
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_clusters(representative.embedding_, iris.labels, method="pca", ax=axes[0],
              title="Iris: reference classes", show=False)
plot_clusters(representative.embedding_, representative.labels_, method="pca", ax=axes[1],
              title="Iris: predicted clusters", show=False)
fig.tight_layout()
fig.savefig(OUTPUT / "iris-label-comparison.png", dpi=150, bbox_inches="tight")
plt.show()

loss_ax = plot_loss_comparison(comparison, run=1, show=False)
loss_ax.figure.savefig(OUTPUT / "feature-losses.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Add one or multiple graph relations per view

These are feature-derived kNN graphs—not independently observed relationships. Each view gets a 5-neighbor and a 10-neighbor relation. The dataset's graph-fusion workflow combines relations before a method consumes them. Feature-only and graph-based experiments should be interpreted as different input settings.

In [ ]:
def attributed_version(data):
    relations = []
    for view in data.Xs:
        graphs = []
        for neighbors in (5, 10):
            adjacency = kneighbors_graph(view, n_neighbors=neighbors, mode="connectivity",
                                        include_self=False)
            graphs.append(adjacency.maximum(adjacency.T))
        relations.append(graphs)
    return MultiViewDataset(
        data.Xs, As=relations, labels=data.labels, name=data.name + "-graphs",
        view_names=data.view_names, modalities=data.modalities,
    )

graph_datasets = {name + "-graphs": attributed_version(data) for name, data in datasets.items()}
graph_comparison = Experiment(
    datasets=graph_datasets,
    models=[LMGEC(n_clusters=3, embedding_dim=2, max_iter=15,
                  random_state=SEED, device="cpu")],
    metrics=metrics, runs=RUNS, save_path=str(OUTPUT / "graph-comparison"),
).run()
print(graph_comparison.results_tables())
plot_loss_comparison([comparison, graph_comparison], run=1)

## 5. Search exact candidates or continuous ranges

Grid evaluates every combination; random selects candidate combinations; Optuna proposes values adaptively. All three use mean NMI across repeats here. This is exploratory tuning on a benchmark—not an unbiased held-out performance estimate. To work without labels, set benchmark=False and choose an internal metric/objective, as illustrated in notebook 05.

In [ ]:
candidates = {
    "n_clusters": [3], "embedding_dim": [2], "max_iter": [15], "device": ["cpu"],
    "temperature": [0.5, 1.0, 2.0],
}
common = dict(
    model_class=LMGEC, dataset=iris, metrics=metrics, best_metric="nmi_score",
    seed=SEED, runs=RUNS,
)
grid = GridSearchExperiment(
    param_grid=candidates, save_path=str(OUTPUT / "grid"), **common,
).run()
random = RandomSearchExperiment(
    param_distributions=candidates, n_iter=3, save_path=str(OUTPUT / "random"), **common,
).run()
optuna = OptunaSearchExperiment(
    param_distributions={
        **candidates, "temperature": {"low": 0.1, "high": 3.0, "log": True},
    },
    n_iter=5, save_path=str(OUTPUT / "optuna"), **common,
).run()
pd.DataFrame([
    {"strategy": name, "best mean NMI": result.best_score, **result.best_params}
    for name, result in [("grid", grid), ("random", random), ("optuna", optuna)]
])

## 6. Reuse the selected model and export

The selected parameters can be passed to the same public estimator class. Repeat the selected configuration and keep the search report alongside the experiment report. JSON/CSV ZIP files contain the recorded results; PNG files above are saved separately.

In [ ]:
best_parameters = {**optuna.best_params, "random_state": SEED}
selected = Experiment(
    datasets=[iris], models=[LMGEC(**best_parameters)], metrics=metrics,
    runs=3, save_path=str(OUTPUT / "selected-model"),
).run()
print(selected.results_tables())

for name, result in [
    ("features", comparison), ("graphs", graph_comparison), ("grid", grid),
    ("random", random), ("optuna", optuna), ("selected", selected),
]:
    result.export(OUTPUT / f"{name}.zip")
print("Reports and plots:", OUTPUT.resolve())

## Next steps

- Change the model list using the method catalog and the capability table.
- Replace the feature matrices with your own aligned views, or use `load_data("acm")`.
- Keep true labels out of model fitting; use them only in benchmark evaluation.
- Use notebook 04 for all registered methods, single-view data and missing-view examples.
- Use notebook 05 for further search objectives and range types.
- Run `polyclustx-display` from a terminal for the same workflows in the graphical interface.